In [1]:
!pip install pdfplumber faiss-cpu sentence-transformers transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 46.0 MB/s eta 0:00:00


In [2]:
from google.colab import files
uploaded = files.upload()

Saving cloud_engineer_resume.pdf to cloud_engineer_resume.pdf
Saving resume_backend_java.pdf to resume_backend_java.pdf
Saving resume_cybersecurity.pdf to resume_cybersecurity.pdf
Saving resume_data_scientist.pdf to resume_data_scientist.pdf
Saving resume_devops.pdf to resume_devops.pdf
Saving resume_frontend.pdf to resume_frontend.pdf
Saving software_developer_resume.pdf to software_developer_resume.pdf


In [3]:
import pdfplumber

def extract_text(file):
    text = ""
    with pdfplumber.open(file) as pdf:
        for page in pdf.pages:
            text += page.extract_text() or ""
    return text

In [4]:
resume_texts = {}

for file_name in uploaded.keys():
    resume_texts[file_name] = extract_text(file_name)

In [5]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-large"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [6]:
def extract_structured_info(text):
    prompt = f"""
Extract structured information from this resume.

Return ONLY in this format:

Skills: ...
Experience: ...
Projects: ...
Certifications: ...

Resume:
{text[:1500]}
"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
    outputs = model.generate(**inputs, max_length=512)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [7]:
structured_data = {}

for file_name, text in resume_texts.items():
    print(f"\n📄 {file_name}")
    print("-" * 50)

    structured = extract_structured_info(text)

    structured_data[file_name] = structured

    print(structured)
    print("\n" + "=" * 80)


📄 cloud_engineer_resume.pdf
--------------------------------------------------
Skills: - Cloud fundamentals - Infrastructure management - Basic networking - Linux basics - Tools / Technologies: - AWS - Azure - EC2 / VM - S3 / Storage - IAM Projects: Deployed a sample web app on cloud VM - Configured cloud storage and access policies


📄 resume_backend_java.pdf
--------------------------------------------------
Skills: 1 Java 2 Spring Boot 3 Microservices 4 REST APIs 5 Hibernate Experience: 4 years experience developing distributed microservices using Spring Boot and REST APIs for scalable backend systems. Certifications: 1 Oracle Certified Java Programmer


📄 resume_cybersecurity.pdf
--------------------------------------------------
Skills: 1 Network Security 2 Penetration Testing 3 SIEM 4 Python Experience Conducted vulnerability assessments and penetration testing to secure enterprise systems. Certifications: 1 Certified Ethical Hacker (CEH)


📄 resume_data_scientist.pdf
----------

In [8]:
cleaned_docs = []

for file_name, content in structured_data.items():
    clean_text = content.replace("\n", " ")

    cleaned_docs.append({
        "file": file_name,
        "text": clean_text
    })

In [9]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
import numpy as np

documents = []
embeddings = []

for doc in cleaned_docs:
    emb = embed_model.encode(doc["text"])

    documents.append(doc["file"])
    embeddings.append(emb)

In [11]:
import faiss

dimension = len(embeddings[0])

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings).astype('float32'))

In [12]:
# def explain_match(jd, resume_text):
#     prompt = f"""
# You are an AI recruiter.

# Job Description:
# {jd}

# Candidate Resume:
# {resume_text}

# Explain:
# - Why this candidate is a good match
# - Matching skills
# - Missing skills

# Keep it short and clear.
# """

#     inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
#     outputs = model.generate(**inputs, max_length=256)

#     return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [20]:
query = input("Enter Job Description: ")

query_embedding = embed_model.encode(query)

D, I = index.search(
    np.array([query_embedding]).astype('float32'),
    k=5
)

print("\n Top Matching resumes:\n")

for rank, (idx, dist) in enumerate(zip(I[0], D[0]), start=1):

    score = 1 / (1 + dist)
    percentage = round(score * 100, 2)

    print(f"Rank {rank}: {documents[idx]}")
    print(f"Match Score: {percentage}%")
    print("-" * 40)

    # explanation = explain_match(query, structured_data[file_name])

    # print("Explanation:")
    # print(explanation)

    # print("-" * 50)

Enter Job Description: We are looking for an engineer who can manage scalable infrastructure in a distributed environment.  The candidate should be comfortable working with virtual machines, cloud storage, identity and access control,  and deploying applications in cloud environments. Experience with platform services and infrastructure automation is a plus.

 Top Matching resumes:

Rank 1: cloud_engineer_resume.pdf
Match Score: 49.130001068115234%
----------------------------------------
Rank 2: resume_devops.pdf
Match Score: 47.5%
----------------------------------------
Rank 3: resume_cybersecurity.pdf
Match Score: 43.849998474121094%
----------------------------------------
Rank 4: resume_backend_java.pdf
Match Score: 43.65999984741211%
----------------------------------------
Rank 5: software_developer_resume.pdf
Match Score: 42.08000183105469%
----------------------------------------
